# Disc Life Inference Notebook

Interactive inference workspace for the **zonal + edge** disc lifing models.

## Supported models
| Model | Folder |
|---|---|
| **PointNetMLPJoint** | `Zonal/Edge_zoneID/PointNetMLPJoint/` |
| **ArGEnT** (cross-att, no SDF) | `Zonal/Edge_zoneID/ArGEnT_self_att_noSDF/` |

## Notebook sections
1. Imports and setup
2. Configure checkpoint paths and model choice
3. Enter geometry deviations
4. Generate geometry and inspect it
5. Run prediction — full-disc life field
6. Compare with nominal geometry
7. Key-location life table
8. Parameter sweep

---
**How to use:**  Run cells top-to-bottom.  Sections 3 and 8 are the main
interactive inputs — edit the offset dict or sweep settings, then re-run
the relevant cells.

## 1 — Imports and setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure the notebook's own folder is on the path
_NB_DIR = Path().resolve()
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

# Helper modules in this folder
from config           import POINTNET_CHECKPOINT, ARGENT_CHECKPOINT, LIFING_MODE, DEFAULT_DEVICE
from inference_helpers import (
    generate_inference_data,
    nominal_offsets,
    summarize_key_life_points,
    get_key_life_points,
    sweep_single_parameter,
)
from model_loading    import load_pointnet_model, load_argent_model, predict_life_field
from plotting_helpers import (
    plot_geometry,
    plot_life_field,
    plot_life_comparison,
    plot_life_diff,
    plot_sweep_results,
    annotate_key_points,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK.  Device:", DEFAULT_DEVICE)

## 2 — Configure checkpoint paths and model choice

Paths are defined in `config.py`.  Edit that file if your checkpoints live elsewhere.
You can also set `ACTIVE_MODEL` to `"argent"` to run that model instead.

In [ ]:
# Choose which model to use in the main prediction cells.
# "pointnet"  → PointNetMLPJoint
# "argent"    → ArGEnT (cross-attention, no SDF)
ACTIVE_MODEL = "pointnet"

# Load both models now so prediction cells are fast.
print("Loading PointNetMLPJoint …")
pn_model,  pn_stats  = load_pointnet_model(POINTNET_CHECKPOINT, device=DEFAULT_DEVICE)
print("  OK —", sum(p.numel() for p in pn_model.parameters()), "parameters")

print("Loading ArGEnT …")
ag_model,  ag_stats  = load_argent_model(ARGENT_CHECKPOINT, device=DEFAULT_DEVICE)
print("  OK —", sum(p.numel() for p in ag_model.parameters()), "parameters")

## 3 — Enter geometry deviations

Specify per-parameter offsets from the nominal geometry (mm).  All entries
default to `0.0` (nominal).  See `Data_gen/config.py` for allowed bounds.

In [ ]:
# Edit the values below.  Set to 0.0 for nominal.
offsets = {
    "bore_radius_inner":       0.0,
    "bore_height":             0.0,
    "bore_thickness":          0.0,
    "lower_transition_height": 0.0,
    "web_height":              0.0,
    "web_thickness":           0.0,
    "upper_transition_height": 0.0,
    "rim_height":              0.0,
    "rim_thickness":           0.0,
    "lower_fillet_radius":     0.0,
    "upper_fillet_radius":     0.0,
}

# Example: reduce lower fillet radius by 0.08 mm
# offsets["lower_fillet_radius"] = -0.08

print("Geometry offsets set.  Non-zero:")
for k, v in offsets.items():
    if v != 0.0:
        print(f"  {k}: {v:+.4f} mm")
if all(v == 0.0 for v in offsets.values()):
    print("  (all nominal)")

## 4 — Generate geometry and inspect it

In [ ]:
print("Generating modified geometry (FEM solve) …")
data_mod = generate_inference_data(offsets, lifing_mode=LIFING_MODE)

print("Generating nominal geometry …")
data_nom = generate_inference_data(nominal_offsets(), lifing_mode=LIFING_MODE)

print(f"  Modified mesh: {data_mod['mesh_nodes'].shape[0]} nodes, "
      f"{data_mod['triangles'].shape[0]} triangles")
print(f"  Edge encoder:  {data_mod['edge_points'].shape[0]} points")
print(f"  Radial breaks: {np.round(data_mod['radial_breaks_mm'], 2)} mm")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_geometry(data_mod, data_nominal=data_nom, ax=axes[0],
              title="Disc geometry (modified vs nominal)")

# Right panel: FEM-computed life for context
plot_life_field(data_mod, data_mod["mesh_fem_life"], ax=axes[1],
                title="FEM life — modified geometry", log_scale=True)

fig.tight_layout()
plt.show()

## 5 — Run prediction — full-disc life field

The model receives the edge geometry (contour points + zone IDs) as its
encoder input and predicts life at every mesh node.

In [ ]:
# Select the active model
if ACTIVE_MODEL == "pointnet":
    model, stats = pn_model, pn_stats
    model_label = "PointNetMLPJoint"
else:
    model, stats = ag_model, ag_stats
    model_label = "ArGEnT"

print(f"Running {model_label} inference on modified geometry …")
life_mod = predict_life_field(model, ACTIVE_MODEL, data_mod, stats, device=DEFAULT_DEVICE)

print(f"Running {model_label} inference on nominal geometry …")
life_nom = predict_life_field(model, ACTIVE_MODEL, data_nom, stats, device=DEFAULT_DEVICE)

print(f"Modified  — min life: {life_mod.min():.3e}  max: {life_mod.max():.3e}")
print(f"Nominal   — min life: {life_nom.min():.3e}  max: {life_nom.max():.3e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Shared scale so both panels are directly comparable
all_life = np.concatenate([life_mod, life_nom])
vmin, vmax = all_life.min(), all_life.max()

plot_life_field(data_mod, life_mod, ax=axes[0],
                title=f"{model_label} — Modified",
                vmin=vmin, vmax=vmax)
plot_life_field(data_nom, life_nom, ax=axes[1],
                title=f"{model_label} — Nominal",
                vmin=vmin, vmax=vmax)

fig.suptitle(f"{model_label}  |  Predicted life field (cycles, log scale)",
             fontsize=11, y=1.01)
fig.tight_layout()
plt.show()

## 6 — Compare with nominal geometry

In [ ]:
# Log-ratio difference map  (positive = modified lives longer)
fig, ax = plt.subplots(figsize=(7, 5))
plot_life_diff(
    data_mod, life_mod, life_nom,
    label_modified=f"Modified ({model_label})",
    label_nominal=f"Nominal ({model_label})",
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: PointNet vs ArGEnT on the same modified geometry
life_mod_pn = predict_life_field(pn_model, "pointnet", data_mod, pn_stats, DEFAULT_DEVICE)
life_mod_ag = predict_life_field(ag_model, "argent",   data_mod, ag_stats, DEFAULT_DEVICE)

fig = plot_life_comparison(
    data_mod, life_mod_pn,
    data_mod, life_mod_ag,
    label_a="PointNetMLPJoint",
    label_b="ArGEnT",
    shared_scale=True,
)
fig.suptitle("Model comparison — modified geometry", fontsize=11, y=1.01)
plt.show()

## 7 — Key-location life table

Scalar life figures at the five canonical disc locations, plus the global
minimum.  Each representative point is the mesh node whose radial coordinate
is nearest the zone midpoint and closest to the disc centre-plane (x ≈ 0).

In [ ]:
rows = []
for (label, life_field, d) in [
    ("PointNet — Modified",  life_mod_pn,  data_mod),
    ("ArGEnT   — Modified",  life_mod_ag,  data_mod),
    ("PointNet — Nominal",
     predict_life_field(pn_model, "pointnet", data_nom, pn_stats, DEFAULT_DEVICE),
     data_nom),
    ("ArGEnT   — Nominal",
     predict_life_field(ag_model, "argent",   data_nom, ag_stats, DEFAULT_DEVICE),
     data_nom),
    ("FEM      — Modified",   data_mod["mesh_fem_life"],  data_mod),
    ("FEM      — Nominal",    data_nom["mesh_fem_life"],  data_nom),
]:
    s = summarize_key_life_points(life_field, d["mesh_nodes"], d["radial_breaks_mm"])
    s["Source"] = label
    rows.append(s)

df_table = pd.DataFrame(rows).set_index("Source")

# Format nicely
cols_life = ["bore", "lower_transition", "web", "upper_transition", "rim", "global_min_life"]
cols_loc  = ["global_min_r", "global_min_x"]

print("Life (cycles) at key disc locations:")
display_df = df_table[cols_life].copy()
display_df.columns = ["Bore", "Lower trans.", "Web", "Upper trans.", "Rim", "Global min"]
display_df.style.format("{:.3e}").set_caption("Predicted life (cycles)")

In [ ]:
# Annotate key points on the modified-geometry life field
key_idx = get_key_life_points(data_mod["mesh_nodes"], data_mod["radial_breaks_mm"])

fig, ax = plt.subplots(figsize=(7, 5))
plot_life_field(data_mod, life_mod_pn, ax=ax,
                title=f"PointNetMLPJoint — Modified geometry\n(annotated key locations)")
annotate_key_points(ax, key_idx, data_mod["mesh_nodes"], life_mod_pn)
plt.tight_layout()
plt.show()

## 8 — Parameter sweep

Vary one offset parameter across a range and track life at each key
location for both models.

In [ ]:
# ── Sweep configuration ────────────────────────────────────────────────────
SWEEP_PARAM  = "lower_fillet_radius"   # parameter to sweep
SWEEP_VALUES = np.linspace(-0.08, 0.08, 9)  # offset range (mm)

# All other parameters held at nominal (zero offset)
SWEEP_FIXED  = nominal_offsets()
# ──────────────────────────────────────────────────────────────────────────

# Build a predict function that returns life fields for both models at once
def _predict_both(data):
    return {
        "PointNet": predict_life_field(pn_model, "pointnet", data, pn_stats, DEFAULT_DEVICE),
        "ArGEnT":   predict_life_field(ag_model, "argent",   data, ag_stats, DEFAULT_DEVICE),
    }

print(f"Running sweep: {SWEEP_PARAM} over {len(SWEEP_VALUES)} values …")
sweep_df = sweep_single_parameter(
    param_name=SWEEP_PARAM,
    sweep_values=SWEEP_VALUES,
    predict_fn=_predict_both,
    fixed_offsets=SWEEP_FIXED,
    lifing_mode=LIFING_MODE,
    verbose=True,
)
print("Sweep complete.")
sweep_df.head()

In [ ]:
fig = plot_sweep_results(
    sweep_df,
    param_name=SWEEP_PARAM,
    locations=["bore", "lower_transition", "web", "upper_transition", "rim"],
    figsize=(10, 5),
    log_y=True,
)
plt.show()

In [ ]:
# Compact sweep summary table
summary_cols = [SWEEP_PARAM, "model",
                "bore", "lower_transition", "web",
                "upper_transition", "rim", "global_min_life"]
sweep_df[summary_cols].style.format(
    {c: "{:.3e}" for c in summary_cols if c not in (SWEEP_PARAM, "model")}
).set_caption(f"Sweep: {SWEEP_PARAM}")